In [8]:
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer

# 1. DEFINE TASKS (KM, KU, KA, KC)
evaluation_tasks = ["1-1 (KM)", "1-3 (KM)", "2-4 (KU)", "3-1 (KA)", "4-1 (KC)"]

# 2. RAW DATA: Reference for KC Level
reference_R = "On May 1, 2015, Marilyn Mosby announced her office had filed charges against six police officers."

# 3. THREE MODEL OUTPUTS
model_outputs = {
    "LLM_A": {
        "1-1": 0.90, "1-3": 0.85, "2-4": 0.88, "3-1": 0.92,
        "4-1": {"T": "Mosby filed charges against 6 officers.", "Tk": "Marilyn Mosby filed charges against 6 officers."}
    },
    "LLM_C": { 
        "1-1": 0.65, "1-3": 0.50, "2-4": 0.60, "3-1": 0.55,
        "4-1": {"T": "The police were charged in Baltimore.", "Tk": "Mosby announced charges for six officers."}
    },
    "LLM_B": {
        "1-1": 0.30, "1-3": 0.15, "2-4": 0.35, "3-1": 0.20,
        "4-1": {"T": "Officers went to court later.", "Tk": "Mosby talked about the police case."}
    }
}

# 4. KOLA METRICS: Self-Contrast and Standardization
def calculate_kc_score(output_dict, reference):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    T, Tk = output_dict['T'], output_dict['Tk']
    d_TR = scorer.score(reference, T)['rougeL'].fmeasure
    d_TTk = scorer.score(Tk, T)['rougeL'].fmeasure # The key Self-Contrast metric 
    d_TkR = scorer.score(reference, Tk)['rougeL'].fmeasure
    return (d_TR + d_TTk + d_TkR) / 3 

def standardize_results(raw_matrix):
    #zij = (xij - mean) / std_dev [2]
    means = raw_matrix.mean(axis=0)
    stds = raw_matrix.std(axis=0).replace(0, 1)
    z_scores = (raw_matrix - means) / stds
    #sij = 100 * (zij - min(z)) / (max(z) - min(z)) 
    min_z = z_scores.min().min()
    max_z = z_scores.max().max()
    return 100 * (z_scores - min_z) / (max_z - min_z)

# 5. EXECUTION
raw_results = []
for m, results in model_outputs.items():
    row = [results["1-1"], results["1-3"], results["2-4"], results["3-1"],
           calculate_kc_score(results["4-1"], reference_R)]
    raw_results.append(row)

df_raw = pd.DataFrame(raw_results, index=model_outputs.keys(), columns=evaluation_tasks)
df_kola = standardize_results(df_raw)

print("--- RAW SCORES ---")
print(df_raw.round(3))
print("\n--- STANDARDIZED KOLA LEADERBOARD (0-100) ---")
print(df_kola.round(1))

--- RAW SCORES ---
       1-1 (KM)  1-3 (KM)  2-4 (KU)  3-1 (KA)  4-1 (KC)
LLM_A      0.90      0.85      0.88      0.92     0.633
LLM_C      0.65      0.50      0.60      0.55     0.237
LLM_B      0.30      0.15      0.35      0.20     0.092

--- STANDARDIZED KOLA LEADERBOARD (0-100) ---
       1-1 (KM)  1-3 (KM)  2-4 (KU)  3-1 (KA)  4-1 (KC)
LLM_A      91.9      94.7      95.5      95.1     100.0
LLM_C      53.6      48.5      46.8      47.7      34.7
LLM_B       0.0       2.3       3.2       2.8      10.8
